In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

import pyrepseq.plotting as pp

plt.style.use('bmh')

In [ ]:
df = pd.read_csv('../data/airr_overlap.csv', index_col=0)
df = df[df['n_repertoire']>1]
df.head()

In [ ]:
df[df['algorithm']=='symscan'].groupby(by=["algorithm","n_repertoire","distance","measure"]).agg(['mean','size'])

In [ ]:
algorithms = ['compairr', 'symscan', 'xt_streaming']
algorithm_labels = {
                    'compairr' : 'CompAIRR',
                    'symscan' : 'SymScan',
                    'xt_streaming' : 'XTNeighbor'
                   }

In [ ]:
fig, axes_arr = plt.subplots(figsize=(6.8, 4.8), ncols=2, nrows=2, sharey=True, sharex=True)
axes = axes_arr.flatten()
for d in [1, 2]:
    for index, algorithm in enumerate(algorithms):
        for midx, measure in enumerate(['hamming', 'leven']):
            data = df[(df['distance']==d) & (df['algorithm']==algorithm) & (df['measure']==measure)]
            mean = data.groupby('n_repertoire').mean(numeric_only=True)
            x, y = mean.index, mean['runtime']

            slope, intercept, r, p, se = scipy.stats.linregress(np.log(x[-3:]), np.log(y[-3:]))
            print(algorithm, d, f'{slope:.3}, {se:.1}')
            l, = axes_arr[d-1, midx].plot(x, y, 'o',
                    label=algorithm_labels[algorithm],
                    color=f'C{6-index}')
            axes_arr[d-1, midx].plot(x, np.exp(slope*np.log(x)+intercept), '-', color=l.get_color())
    axes_arr[d-1, 0].set_yscale('log')
    axes_arr[d-1, 0].set_xscale('log', base=2)
axes_arr[0, 0].set_title('Hamming distance', fontsize='medium')
axes_arr[0, 1].set_title('Levenshtein distance', fontsize='medium')
axes_arr[0, 1].text(1.05, 0.5, 'Distance = 1', transform=axes_arr[0, 1].transAxes,
                    rotation=90, va='center', fontweight='bold')
axes_arr[1, 1].text(1.05, 0.5, 'Distance = 2', transform=axes_arr[1, 1].transAxes,
                    rotation=90, va='center', fontweight='bold')
for i in range(2):
    axes_arr[1, i].set_xlabel('# Repertoires')
    axes_arr[i, 0].set_ylabel('Time in seconds')
legend = axes_arr[0, -1].legend(bbox_to_anchor=(1.2, 0.9), loc='upper left')
legend_texts = legend.get_texts()
for i in range(1, 3):
    legend_texts[-i].set_weight('bold') 
axes_arr[0, 0].text(-0.25, 1.0, 'A', transform=axes_arr[0, 0].transAxes, fontweight="bold", va="top")
fig.tight_layout(w_pad=2, pad=1.0)
axes_arr[0, 1].text(-0.08, 1.0, 'B', transform=axes_arr[0, 1].transAxes, fontweight="bold", va="top")
axes_arr[1, 0].text(-0.25, 1.0, 'C', transform=axes_arr[1, 0].transAxes, fontweight="bold", va="top")
axes_arr[1, 1].text(-0.08, 1.0, 'D', transform=axes_arr[1, 1].transAxes, fontweight="bold", va="top")
fig.savefig('figs/airr_overlap.pdf')